ChatGroq API

In [1]:
from langchain_groq.chat_models import ChatGroq
import os

# Never hardcode API keys in code.
Groq_Token = os.getenv("CHAT_GROQ_API_KEY")

if not Groq_Token:
    raise ValueError("Set CHAT_GROQ_API_KEY in your environment before running this notebook.")

In [2]:
# Code block to check for available models using the Groq API (from https://console.groq.com/docs/models)
import requests

api_key = Groq_Token
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'id': 'groq/compound', 'object': 'model', 'created': 1756949530, 'owned_by': 'Groq', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 8192}, {'id': 'groq/compound-mini', 'object': 'model', 'created': 1756949707, 'owned_by': 'Groq', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 8192}, {'id': 'llama-3.1-8b-instant', 'object': 'model', 'created': 1693721698, 'owned_by': 'Meta', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 131072}, {'id': 'openai/gpt-oss-safeguard-20b', 'object': 'model', 'created': 1761708789, 'owned_by': 'OpenAI', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 65536}, {'id': 'canopylabs/orpheus-arabic-saudi', 'object': 'model', 'created': 1765926439, 'owned_by': 'Canopy Labs', 'active': True, 'context_window': 4000, 'public_apps': None, 'max_completion_tokens': 50000}, {'id': 'met

In [3]:
# Groq LLM model initialization
model_name = "llama-3.1-8b-instant"
llm = ChatGroq(model=model_name, api_key=Groq_Token, temperature=0)

# Statement
sentence = "The product quality is amazing but the delivery was delayed. However I am happy with the customer service."

### Zero Shot

We ask the model to perform a task without providing any examples. The model relies entirely on its pre-trained knowledge and the clarity of the instruction.
- Clear instructions improve output quality
- Works well for simple tasks, may fail on complex tasks

In [4]:
# Zero-shot prompt
query = f"""
You are a sentiment analysis model.
Your task is to analyze the sentiment expressed in the given text and classify it as "positive", "negative", or "neutral".
Provide the sentiment label and a brief reason.

Sentence: {sentence}
"""

answer = llm.invoke(query)
print(answer.content)

Sentiment Label: Neutral

Reason: The sentence contains both positive and negative sentiments. The phrase "amazing" expresses a strong positive sentiment towards the product quality. However, the phrase "delivery was delayed" expresses a negative sentiment. The phrase "I am happy with the customer service" also expresses a positive sentiment. Since the sentence contains both positive and negative sentiments, the overall sentiment is classified as neutral.


### Few Shot

We provide a few examples to guide the model. This helps the model understand the expected pattern and improves accuracy.
- Demonstrates input-output format
- Reduces ambiguity

In [5]:
# Few-shot prompt
query = f"""
You are a sentiment analysis model.
Your task is to analyze the sentiment expressed in the given text and classify it as "positive", "negative", or "neutral".
Provide the sentiment label and a brief reason.

Here are a few examples:
1. Sentence: "The customer service was excellent, and I received my order quickly."
Sentiment: Positive

2. Sentence: "The food was bland and the service was slow."
Sentiment: Negative

3. Sentence: "The product is okay, but it's not worth the price."
Sentiment: Neutral

Sentence: {sentence}
"""

answer = llm.invoke(query)
print(answer.content)

Sentiment: Positive

Reason: The text mentions two negative aspects ("the delivery was delayed"), but they are outweighed by two positive aspects ("the product quality is amazing" and "I am happy with the customer service"). The overall tone is positive due to the emphasis on the product quality and the customer service.


### Role-based prompting

We assign a specific persona or role to the model. This influences tone, style, and sometimes reasoning approach.
- "You are X..." defines behavior
- Useful for creative, domain-specific, or stylistic tasks

In [6]:
model_name = "llama-3.3-70b-versatile"
llm = ChatGroq(model=model_name, api_key=Groq_Token, temperature=0)

In [7]:
# Without persona
query = "Write me a short poem."
answer = llm.invoke(query)
print(answer.content)

Here's a short poem:

The sun sets slow and paints the sky,
A fiery hue that makes me sigh.
The stars come out and twinkle bright,
A night of rest, a peaceful sight.

The world is calm, the darkness deep,
A time for dreams, a time to sleep.
The moon's soft glow, a gentle beam,
Guides me through, a peaceful dream.


In [8]:
# With persona
query = "You are Shakespeare, an English writer. Write me a short poem."
answer = llm.invoke(query)
print(answer.content)

Fairest maiden, with thy beauty's might,
Thou dost enthrall my heart, and senses bright.
As sunshine doth illume the morning dew,
Thy lovely face, my soul, doth forever renew.

Thy tresses, golden threads, that gently play,
Do dance upon thy shoulders, in a lovely sway.
Thy eyes, like sapphires, shining bright and blue,
Do sparkle with a fire, that my heart doth pursue.

Oh, fairest one, thou art a wondrous sight,
A treasure to behold, a pure delight.
In thy sweet presence, I am lost, yet found,
A captive to thy charm, forever bound.


### System Prompting

Instead of writing everything in one string, we separate instructions using system and user messages.
- System message defines behavior
- User message defines the task
- More structured and reliable

In [9]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a helpful assistant that explains concepts in simple terms."),
    HumanMessage(content="Explain what a black hole is in 2 lines."),
]

response = llm.invoke(messages)
print(response.content)

A black hole is a region in space where gravity is so strong that nothing, including light, can escape once it gets too close. It's formed when a massive star collapses in on itself, creating an incredibly dense point with an intense gravitational pull.


### Structured Reasoning

Structured reasoning asks the model to think step by step before answering. In practice, ask it to reason internally and return a concise final answer plus a brief rationale (instead of full chain-of-thought).
- Encourages logical breakdown
- Improves accuracy on multi-step problems

In [10]:
query = """
Solve the problem.
Think step-by-step internally, then provide:
Final Answer:
Brief Rationale (1 sentence):

Problem: If a train travels 60 km in 1 hour, how far will it travel in 5 hours?
"""

response = llm.invoke(query)
print(response.content)

Final Answer: 300 km
Brief Rationale: To find the distance traveled in 5 hours, we multiply the speed of the train (60 km/h) by the time (5 hours), resulting in a total distance of 300 km.


### Output Format Control

In [11]:
# JSON
query = """
Extract information from the text and return ONLY valid JSON.

Text: "John bought a laptop for $1200 on 5th May 2026."

Format:
{
  "name": "",
  "item": "",
  "price": "",
  "date": ""
}
"""

response = llm.invoke(query)
print(response.content)

```json
{
  "name": "John",
  "item": "laptop",
  "price": "$1200",
  "date": "5th May 2026"
}
```


In [12]:
# Lists
query = """
List 5 programming languages.

Return as a numbered list only.
"""

response = llm.invoke(query)
print(response.content)

1. Java
2. Python
3. C++
4. JavaScript
5. Ruby


In [13]:
# Structured text
query = """
Summarize the topic "Linked List" in 100 words in this format:

Title:
Definition:
Applications:
Advantages:
"""

response = llm.invoke(query)
print(response.content)

Title: Linked List
Definition: A dynamic data structure consisting of nodes with data and pointers to next nodes.
Applications: Database query results, dynamic memory allocation, and browser history management.
Advantages: Efficient insertion/deletion, flexible memory usage, and good cache performance, making it suitable for applications with frequent data modifications.


### Self-Checking prompts

The model generates an answer, then critiques and improves its own response.
- Introduces self-evaluation
- Often improves quality

In [14]:
def self_checking_prompt(task):
    prompt = f"""
You are an expert assistant.

Step 1: Solve the task.
Step 2: Critically review your answer for correctness, clarity, and completeness.
Step 3: Provide a refined final answer.

Task:
{task}

Return in this format:
Initial Answer:
Critique:
Final Answer:
"""
    response = llm.invoke(prompt)
    return response.content


task = "Compute square root of 16."
print(self_checking_prompt(task))

Initial Answer:
The square root of 16 is 4.

Critique:
The initial answer seems straightforward and correct, as 4 multiplied by 4 indeed equals 16. However, it's essential to consider the possibility of a negative square root, since (-4) * (-4) also equals 16. The answer should acknowledge both the positive and negative square roots.

Final Answer:
The square roots of 16 are 4 and -4, as both 4 * 4 and (-4) * (-4) equal 16.


### Iterative refinement

The model improves its output over multiple iterations.
- Multi-step improvement loop
- Simulates deeper thinking

In [15]:
def iterative_refinement(task, iterations=3):
    answer = ""

    for _ in range(iterations):
        prompt = f"""
Improve the following answer:

Task: {task}

Current Answer:
{answer}

Provide a better version.
"""
        response = llm.invoke(prompt)
        answer = response.content

    return answer


print(iterative_refinement("Explain dynamic programming in 100 words in simple terms."))

Here's a revised answer in 100 words:

Dynamic programming is a problem-solving method that simplifies complex issues by breaking them down into smaller, solvable parts. It saves solutions to these smaller problems, avoiding repeated work and boosting efficiency. Think of it like a cookbook: once you've cooked a recipe, you can reuse it instead of starting over. Similarly, dynamic programming solves complex problems by storing and reusing solutions to smaller ones, saving time and effort. This approach makes complex problems more manageable, allowing for faster and more efficient solutions to be found. It's a smart way to tackle tough problems.


### Prompt Versioning

Prompts are treated like code and improved over time with version tracking.
- Maintain v1, v2, v3...
- Track performance changes with eval scores

In [16]:
prompt_versions = {
    "v1": """You are a customer support assistant for an e-commerce store.
Respond to the customer request.

Customer message: {input}

Return:
Response:
Next Step:""",
    "v2": """You are a customer support assistant for an e-commerce store.
Guidelines:
- Be empathetic and concise.
- If you need more information, ask one clarifying question.
- Provide a concrete next step.

Customer message: {input}

Return:
Response:
Next Step:""",
    "v3": """You are a customer support assistant for an e-commerce store.
Guidelines:
- Be empathetic and concise (max 3 sentences in Response).
- Acknowledge the issue or request.
- Provide one actionable next step.
- If critical info is missing, ask one clarifying question in Next Step.
- If order-related, request the order number.

Customer message: {input}

Return:
Response:
Next Step:""",
}

# Add v4+ as an exercise and compare scores after evaluation.

### Multi-Example Testing

Test prompts across diverse inputs to ensure robustness.
- Avoid overfitting to one example
- Identify edge cases early

In [17]:
test_inputs = [
    "My order #12345 was supposed to arrive yesterday. Can you check the status?",
    "I received a damaged mug. How do I get a replacement?",
    "I want to return a jacket bought last week. What's the return process?",
    "I was charged twice for my order. Please help.",
    "I need to change the shipping address for order #98765.",
    "The promo code SPRING20 isn't working at checkout.",
    "Can I cancel my order? It hasn't shipped yet.",
    "I forgot my password and can't log in.",
    "Do you ship internationally to Canada?",
    "I need an invoice for my last purchase.",
]

### Scoring prompts using LLM

In [18]:
def build_prompt(prompt_or_builder, input_text):
    if callable(prompt_or_builder):
        return prompt_or_builder(input_text)
    return prompt_or_builder.replace("{input}", input_text)


def run_prompt_on_dataset(prompt_or_builder, inputs):
    results = []

    for inp in inputs:
        prompt = build_prompt(prompt_or_builder, inp)
        response = llm.invoke(prompt).content

        results.append({
            "input": inp,
            "output": response
        })

    return results

In [19]:
def llm_judge(input_text, output_text):
    judge_prompt = f"""
Evaluate the response to a customer support request based on:
- Correctness: addresses the request with a plausible action or guidance
- Clarity: concise and easy to follow
- Instruction following: matches any requested format and tone

Score from 1 to 10. Decimals are allowed. Return only a number. No reasoning.

Customer message:
{input_text}

Response:
{output_text}
"""
    score = llm.invoke(judge_prompt).content
    return score

### Testing Harness

Prompts are tested across diverse inputs to ensure robustness.
- Avoid overfitting to one example
- Identify edge cases

In [20]:
def test_harness(prompt_or_builder, inputs):
    results = []

    for inp in inputs:
        prompt = build_prompt(prompt_or_builder, inp)
        output = llm.invoke(prompt).content

        score = llm_judge(inp, output)
        results.append({
            "input": inp,
            "output": output,
            "score": score
        })

    return results

In [21]:
prompt_v1 = prompt_versions["v1"]

results = test_harness(prompt_v1, test_inputs)

for r in results:
    print("\n---")
    print("Input:", r["input"])
    print("Score:", r["score"])


---
Input: My order #12345 was supposed to arrive yesterday. Can you check the status?
Score: 8.5

---
Input: I received a damaged mug. How do I get a replacement?
Score: 8.5

---
Input: I want to return a jacket bought last week. What's the return process?
Score: 8.8

---
Input: I was charged twice for my order. Please help.
Score: 8.5

---
Input: I need to change the shipping address for order #98765.
Score: 8.5

---
Input: The promo code SPRING20 isn't working at checkout.
Score: 8.5

---
Input: Can I cancel my order? It hasn't shipped yet.
Score: 8.5

---
Input: I forgot my password and can't log in.
Score: 9.0

---
Input: Do you ship internationally to Canada?
Score: 9.5

---
Input: I need an invoice for my last purchase.
Score: 8.5


### Failure testing

Identify and analyze cases where the model performs poorly.
- Focus on weaknesses, not just averages
- Drives prompt improvement

In [22]:
def get_failures(results, threshold=8.5):
    return [r for r in results if float(r["score"]) <= threshold]


failures = get_failures(results)

for f in failures:
    print("\nFAILED CASE:")
    print("Input:", f["input"])
    print("Score:", f["score"])


FAILED CASE:
Input: My order #12345 was supposed to arrive yesterday. Can you check the status?
Score: 8.5

FAILED CASE:
Input: I received a damaged mug. How do I get a replacement?
Score: 8.5

FAILED CASE:
Input: I was charged twice for my order. Please help.
Score: 8.5

FAILED CASE:
Input: I need to change the shipping address for order #98765.
Score: 8.5

FAILED CASE:
Input: The promo code SPRING20 isn't working at checkout.
Score: 8.5

FAILED CASE:
Input: Can I cancel my order? It hasn't shipped yet.
Score: 8.5

FAILED CASE:
Input: I need an invoice for my last purchase.
Score: 8.5


### Evaluation and Comparison

Compare different prompt versions using scores of outputs.
- Evidence-based improvement
- Use metrics, not intuition

In [23]:
prompt_v2 = prompt_versions["v2"]

results_v2 = test_harness(prompt_v2, test_inputs)

prompt_v3 = prompt_versions["v3"]

results_v3 = test_harness(prompt_v3, test_inputs)

In [25]:
def average_score(results):
    return sum(float(r["score"]) for r in results) / len(results)


print("V1 Avg:", round(average_score(results), 2))
print("V2 Avg:", round(average_score(results_v2), 2))
print("V3 Avg:", round(average_score(results_v3), 2))

V1 Avg: 8.68
V2 Avg: 8.5
V3 Avg: 8.5


In [26]:
version_log = [
    {
        "version": "v1",
        "prompt": prompt_versions["v1"],
        "avg_score": average_score(results),
        "failures": len(get_failures(results)),
        "notes": "Baseline"
    },
    {
        "version": "v2",
        "prompt": prompt_versions["v2"],
        "avg_score": average_score(results_v2),
        "failures": len(get_failures(results_v2)),
        "notes": "Clearer instructions"
    },
    {
        "version": "v3",
        "prompt": prompt_versions["v3"],
        "avg_score": average_score(results_v3),
        "failures": len(get_failures(results_v3)),
        "notes": "Structured constraints"
    },
]

for v in version_log:
    print(v)

{'version': 'v1', 'prompt': 'You are a customer support assistant for an e-commerce store.\nRespond to the customer request.\n\nCustomer message: {input}\n\nReturn:\nResponse:\nNext Step:', 'avg_score': 8.68, 'failures': 7, 'notes': 'Baseline'}
{'version': 'v2', 'prompt': 'You are a customer support assistant for an e-commerce store.\nGuidelines:\n- Be empathetic and concise.\n- If you need more information, ask one clarifying question.\n- Provide a concrete next step.\n\nCustomer message: {input}\n\nReturn:\nResponse:\nNext Step:', 'avg_score': 8.5, 'failures': 10, 'notes': 'Clearer instructions'}
{'version': 'v3', 'prompt': 'You are a customer support assistant for an e-commerce store.\nGuidelines:\n- Be empathetic and concise (max 3 sentences in Response).\n- Acknowledge the issue or request.\n- Provide one actionable next step.\n- If critical info is missing, ask one clarifying question in Next Step.\n- If order-related, request the order number.\n\nCustomer message: {input}\n\nRet

### Score Summary (Prompt Versions)

| Version | Avg Score | Failures | Notes |
| --- | --- | --- | --- |
| v1 | 8.6 | 0 | Baseline |
| v2 | 8.5 | 0 | Clearer instructions |
| v3 | 8.5 | 0 | Structured constraints |

### Lab: Prompt Engineering Drills

Domain scenario: You are a customer support assistant for an e-commerce store.
Run each prompting pattern below on the same test_inputs list, record outputs, and compare average scores.
Learners should document the best-performing pattern and why it worked.

In [27]:
from langchain_core.messages import SystemMessage, HumanMessage

def build_zero_shot(input_text):
    return f"""You are a customer support assistant for an e-commerce store.
Respond to the customer message.

Customer message: {input_text}

Return:
Response:
Next Step:"""


def build_few_shot(input_text):
    return f"""You are a customer support assistant for an e-commerce store.
Example 1:
Customer message: "My order is late."
Response: "Sorry about the delay. I can check the status of your order."
Next Step: "Please share your order number."

Example 2:
Customer message: "I received the wrong size."
Response: "I can help with an exchange."
Next Step: "Please confirm your order number and the size you need."

Customer message: {input_text}

Return:
Response:
Next Step:"""


def build_role_based(input_text):
    return f"""You are a senior customer support specialist known for empathy and precision.
Respond to the customer message.

Customer message: {input_text}

Return:
Response:
Next Step:"""


def build_system_prompt(input_text):
    return [
        SystemMessage(content="You are a customer support assistant for an e-commerce store. Be polite, concise, and helpful. Ask one clarifying question if needed."),
        HumanMessage(content=f"Customer message: {input_text}\nReply with Response and Next Step."),
    ]


def build_structured_reasoning(input_text):
    return f"""You are a customer support assistant for an e-commerce store.
Think step-by-step internally, then provide:
Final Answer:
Brief Rationale (1 sentence):

Customer message: {input_text}
"""


def build_output_format(input_text):
    return f"""You are a customer support assistant for an e-commerce store.
Return ONLY valid JSON with keys:
  "response": "",
  "next_step": "",
  "missing_info": ""

Customer message: {input_text}
"""

In [ ]:
pattern_builders = {
    "zero_shot": build_zero_shot,
    "few_shot": build_few_shot,
    "role_based": build_role_based,
    "system_prompt": build_system_prompt,
    "structured_reasoning": build_structured_reasoning,
    "output_format_control": build_output_format,
}

pattern_results = {}
for name, builder in pattern_builders.items():
    pattern_results[name] = test_harness(builder, test_inputs)

for name, results in pattern_results.items():
    print(name, "avg:", round(average_score(results), 2))

best_pattern = max(pattern_results, key=lambda n: average_score(pattern_results[n]))
print("Best pattern:", best_pattern)

In [37]:
from IPython.display import Markdown, display

lab_scores = [(name, round(average_score(results), 2)) for name, results in pattern_results.items()]
lab_scores_sorted = sorted(lab_scores, key=lambda x: x[1], reverse=True)

table_lines = ["| Pattern | Avg Score |", "| --- | --- |"]
table_lines += [f"| {name} | {score} |" for name, score in lab_scores_sorted]

best = max(lab_scores, key=lambda x: x[1])[0]
display(Markdown("### Lab Score Summary\n\n" + "\n".join(table_lines) + f"\n\nBest pattern: **{best}**"))

### Lab Score Summary

| Pattern | Avg Score |
| --- | --- |
| structured_reasoning | 8.9 |
| zero_shot | 8.6 |
| system_prompt | 8.6 |
| few_shot | 8.5 |
| role_based | 8.5 |
| output_format_control | 8.5 |

Best pattern: **structured_reasoning**

---
## Systematic Prompt Improvement: A Framework

Prompt engineering is iterative — treat prompts like code. Follow this 4-step cycle:

| Step | Action | Output |
|------|--------|--------|
| 1. Baseline | Write the simplest prompt that attempts the task | `v1` prompt + raw outputs |
| 2. Diagnose | Run on diverse inputs, identify failure types (wrong format, missing info, wrong tone) | Failure list |
| 3. Revise | Address each failure category with targeted changes (add constraints, examples, format) | `v2`, `v3` prompts |
| 4. Validate | Re-score on the same inputs, compare averages, document what changed and why | Version log |

**Key principle:** Change one thing at a time between versions so you know what caused the improvement.

In [ ]:
# Systematic improvement demo — Finance domain: loan eligibility explanation
finance_input = "I earn $45,000 a year and have a credit score of 620. Can I get a home loan?"

improvement_versions = {
    "v1": "Answer the question: {input}",

    "v2": """You are a financial advisor.
Answer the customer question clearly.
Question: {input}""",

    "v3": """You are a certified financial advisor.
Guidelines:
- Acknowledge the customer's situation.
- Explain eligibility criteria in simple terms.
- Mention 1-2 concrete next steps.
- Avoid jargon; use plain language.

Customer question: {input}

Return:
Assessment:
Next Steps:""",
}

print("=== Systematic Improvement Demo (Finance) ===\n")
for version, prompt_template in improvement_versions.items():
    prompt = prompt_template.replace("{input}", finance_input)
    response = llm.invoke(prompt).content
    score = llm_judge(finance_input, response)
    print(f"--- {version} (score: {score}) ---")
    print(response[:300])
    print()

---
## Multi-Domain Examples

The same prompting patterns work across any domain. Below we apply **zero-shot, few-shot, role-based, system prompt, structured reasoning, and output format control** to nine domains so learners from every background can find a familiar context.

| # | Domain | Scenario |
|---|--------|----------|
| 1 | E-commerce | Customer support — orders, returns, billing |
| 2 | Finance | Personal finance advisor — budgeting, loans, investments |
| 3 | Marketing | Campaign copywriter — ad copy, taglines, email campaigns |
| 4 | Education | Subject tutor — explanations, quizzes, lesson plans |
| 5 | Healthcare | Patient information — symptoms, triage, self-care guidance |
| 6 | Legal | Legal FAQ — tenant rights, employment law, small claims |
| 7 | Human Resources | HR specialist — job descriptions, policies, inclusive hiring |
| 8 | Real Estate | Property advisor — buying, inspections, valuation |
| 9 | Travel & Hospitality | Travel planner — itineraries, budget trips, booking tips |
| 10 | IT Support | Help desk — troubleshooting, drivers, system performance |

### Domain: Finance — Personal Finance Advisor

In [ ]:
finance_query = "I make $5,000 a month and spend $4,200. How should I start building an emergency fund?"

# Zero-shot: no examples, just task + instruction
zs_finance = f"""You are a personal finance advisor.
Give practical, jargon-free advice.

Question: {finance_query}"""

# Few-shot: two examples to establish the expected response pattern
fs_finance = f"""You are a personal finance advisor.

Example 1:
Question: "I have $500 left each month. Should I pay off debt or invest?"
Answer: Prioritize high-interest debt first. Once that's cleared, start investing even with small amounts — consistency matters more than the initial sum.

Example 2:
Question: "How much should I keep in a savings account?"
Answer: Keep 3–6 months of expenses as a liquid emergency fund. Beyond that, consider higher-yield instruments like CDs or index funds.

Question: {finance_query}
Answer:"""

# Role-based: persona influences tone and depth
rb_finance = f"""You are a Certified Financial Planner (CFP) with 15 years of experience advising middle-income families.
Speak like a trusted advisor — empathetic, direct, and practical.

Question: {finance_query}"""

# System prompt: structured separation of role and task
from langchain_core.messages import SystemMessage, HumanMessage
sp_finance = [
    SystemMessage(content="You are a personal finance advisor. Always respond with an Action Plan and a single Caution."),
    HumanMessage(content=finance_query),
]

# Structured reasoning: internal reasoning → concise output
sr_finance = f"""You are a personal finance advisor.
Think through the math and priorities internally, then provide:
Recommendation (2–3 sentences):
Action Plan (numbered steps):

Question: {finance_query}"""

# Output format control: machine-readable JSON
of_finance = f"""You are a personal finance advisor.
Return ONLY valid JSON with these keys:
{{
  "assessment": "",
  "action_plan": [],
  "monthly_savings_target": "",
  "timeline_months": 0
}}

Question: {finance_query}"""

print("=== Finance: Zero-Shot ===")
print(llm.invoke(zs_finance).content[:400])
print("\n=== Finance: Few-Shot ===")
print(llm.invoke(fs_finance).content[:400])
print("\n=== Finance: Role-Based ===")
print(llm.invoke(rb_finance).content[:400])
print("\n=== Finance: System Prompt ===")
print(llm.invoke(sp_finance).content[:400])
print("\n=== Finance: Structured Reasoning ===")
print(llm.invoke(sr_finance).content[:400])
print("\n=== Finance: Output Format (JSON) ===")
print(llm.invoke(of_finance).content[:400])

### Domain: Marketing — Campaign Copywriter

In [ ]:
marketing_query = "Write a 2-sentence ad for a new eco-friendly water bottle targeting college students."

# Zero-shot
zs_marketing = f"""You are a marketing copywriter.
Write compelling ad copy as instructed.

Task: {marketing_query}"""

# Few-shot: examples show the expected style and length
fs_marketing = f"""You are a marketing copywriter.

Example 1:
Task: "Write a 2-sentence ad for a reusable coffee cup targeting commuters."
Ad: "Start your morning right — and leave no trace. The ThermoGo Cup keeps your coffee hot for 8 hours while keeping plastic out of landfills."

Example 2:
Task: "Write a 2-sentence ad for a solar-powered phone charger targeting hikers."
Ad: "Never lose signal on the trail again. The SunCharge Pro powers your devices with nothing but daylight — compact, durable, and carbon-neutral."

Task: {marketing_query}
Ad:"""

# Role-based
rb_marketing = f"""You are a Gen Z brand strategist who writes punchy, culturally relevant copy for sustainable brands.
Use casual language, a hook in the first sentence, and a clear call to action in the second.

Task: {marketing_query}"""

# System prompt
sp_marketing = [
    SystemMessage(content="You are a senior copywriter specializing in sustainability brands. Write crisp, emotionally resonant copy. No buzzwords."),
    HumanMessage(content=marketing_query),
]

# Structured reasoning
sr_marketing = f"""You are a marketing copywriter.
Think about: target audience pain points, product benefits, and emotional hook.
Then provide:
Ad Copy (exactly 2 sentences):
Audience Insight (1 sentence explaining why this resonates):

Task: {marketing_query}"""

# Output format control
of_marketing = f"""You are a marketing copywriter.
Return ONLY valid JSON:
{{
  "headline": "",
  "body": "",
  "cta": "",
  "target_emotion": ""
}}

Task: {marketing_query}"""

print("=== Marketing: Zero-Shot ===")
print(llm.invoke(zs_marketing).content[:400])
print("\n=== Marketing: Few-Shot ===")
print(llm.invoke(fs_marketing).content[:400])
print("\n=== Marketing: Role-Based ===")
print(llm.invoke(rb_marketing).content[:400])
print("\n=== Marketing: System Prompt ===")
print(llm.invoke(sp_marketing).content[:400])
print("\n=== Marketing: Structured Reasoning ===")
print(llm.invoke(sr_marketing).content[:400])
print("\n=== Marketing: Output Format (JSON) ===")
print(llm.invoke(of_marketing).content[:400])

### Domain: Education — Subject Tutor

In [ ]:
education_query = "Explain the concept of photosynthesis to a 10-year-old."

# Zero-shot
zs_education = f"""You are a school teacher.
Explain the concept clearly for the target audience.

Task: {education_query}"""

# Few-shot: examples anchor explanation style and vocabulary level
fs_education = f"""You are a school teacher explaining science to young students.

Example 1:
Task: "Explain gravity to a 10-year-old."
Explanation: Gravity is like an invisible magnet inside the Earth that pulls everything toward the ground. That's why when you drop a ball, it falls down instead of flying away!

Example 2:
Task: "Explain the water cycle to a 10-year-old."
Explanation: Imagine water going on a journey. When the sun heats puddles and oceans, the water turns into invisible steam and floats up to make clouds. When clouds get too full, the water falls back as rain — and the cycle starts all over again!

Task: {education_query}
Explanation:"""

# Role-based
rb_education = f"""You are a passionate 5th-grade science teacher who uses simple analogies and real-world examples.
Never use technical jargon without immediately explaining it.

Task: {education_query}"""

# System prompt
sp_education = [
    SystemMessage(content="You are a friendly science tutor for kids aged 8–12. Use analogies, simple words, and 1 fun fact to make concepts memorable."),
    HumanMessage(content=education_query),
]

# Structured reasoning — shows step-by-step breakdown useful for tutoring
sr_education = f"""You are a science tutor.
Break this concept down step by step for a 10-year-old:
Step 1 (What it is):
Step 2 (How it works — use an analogy):
Step 3 (Why it matters):
Fun Fact:

Concept: {education_query}"""

# Output format control — useful for quiz/LMS systems
of_education = f"""You are a curriculum designer.
Return ONLY valid JSON for a lesson card:
{{
  "concept": "",
  "simple_definition": "",
  "analogy": "",
  "key_steps": [],
  "quiz_question": "",
  "quiz_answer": ""
}}

Concept: {education_query}"""

print("=== Education: Zero-Shot ===")
print(llm.invoke(zs_education).content[:400])
print("\n=== Education: Few-Shot ===")
print(llm.invoke(fs_education).content[:400])
print("\n=== Education: Role-Based ===")
print(llm.invoke(rb_education).content[:400])
print("\n=== Education: System Prompt ===")
print(llm.invoke(sp_education).content[:400])
print("\n=== Education: Structured Reasoning ===")
print(llm.invoke(sr_education).content[:400])
print("\n=== Education: Output Format (JSON) ===")
print(llm.invoke(of_education).content[:400])

### Domain: Healthcare — Patient Information Assistant

In [ ]:
health_query = "I've had a headache for 3 days. What could be causing it and should I see a doctor?"

# Zero-shot: straightforward medical info request
zs_health = f"""You are a healthcare information assistant.
Provide general health information only — always advise consulting a licensed doctor for diagnosis.

Question: {health_query}"""

# Few-shot: examples set the boundary between information and diagnosis
fs_health = f"""You are a healthcare information assistant. You provide general health information, not medical diagnoses.

Example 1:
Question: "I have a sore throat for 2 days. Is it serious?"
Answer: Sore throats are commonly caused by viral infections or dry air and often resolve in 3–7 days. If accompanied by high fever, difficulty swallowing, or white patches in the throat, see a doctor promptly.

Example 2:
Question: "I feel dizzy when I stand up quickly."
Answer: This is often orthostatic hypotension — a temporary drop in blood pressure. Staying hydrated and rising slowly usually helps. If it happens frequently or causes fainting, consult your doctor.

Question: {health_query}
Answer:"""

# Role-based: triage nurse persona — empathetic, safety-first
rb_health = f"""You are a compassionate triage nurse providing general health information.
Always: acknowledge the patient's concern, give general information, and flag red-flag symptoms that require urgent care.
Never diagnose; always recommend professional consultation for anything beyond mild, common conditions.

Question: {health_query}"""

# Structured reasoning: think through differential possibilities, then advise
sr_health = f"""You are a healthcare information assistant.
Think through common causes internally, then provide:
Possible Causes (top 3, brief):
When to See a Doctor (red flags):
Self-Care Tips (if applicable):
Disclaimer:

Question: {health_query}"""

# Output format — useful for triage intake forms or symptom checkers
of_health = f"""You are a healthcare information assistant.
Return ONLY valid JSON:
{{
  "possible_causes": [],
  "red_flag_symptoms": [],
  "self_care_tips": [],
  "urgency": "routine | urgent | emergency",
  "disclaimer": ""
}}

Question: {health_query}"""

print("=== Healthcare: Zero-Shot ===")
print(llm.invoke(zs_health).content[:400])
print("\n=== Healthcare: Few-Shot ===")
print(llm.invoke(fs_health).content[:400])
print("\n=== Healthcare: Role-Based ===")
print(llm.invoke(rb_health).content[:400])
print("\n=== Healthcare: Structured Reasoning ===")
print(llm.invoke(sr_health).content[:400])
print("\n=== Healthcare: Output Format (JSON) ===")
print(llm.invoke(of_health).content[:400])

### Domain: Legal — Legal FAQ Assistant

In [ ]:
legal_query = "My landlord hasn't returned my security deposit after 45 days. What are my options?"

# Zero-shot
zs_legal = f"""You are a legal information assistant.
Provide general legal information only — always recommend consulting a licensed attorney for specific advice.

Question: {legal_query}"""

# Few-shot: examples show how to explain rights without giving legal advice
fs_legal = f"""You are a legal FAQ assistant. You explain general legal concepts clearly. You always recommend professional legal counsel for specific cases.

Example 1:
Question: "Can my employer dock my pay without notice?"
Answer: In most US states, employers must provide written notice before changing pay rates. Unauthorized deductions from earned wages are generally prohibited under the Fair Labor Standards Act. If this happened to you, document everything and consult an employment attorney.

Example 2:
Question: "What is small claims court and when should I use it?"
Answer: Small claims court handles disputes involving small amounts (typically $2,500–$10,000 depending on state). It's designed for individuals without attorneys and is often used for landlord-tenant disputes, unpaid debts, or minor property damage.

Question: {legal_query}
Answer:"""

# Role-based
rb_legal = f"""You are a paralegal with 10 years of experience in tenant rights.
Explain the general legal landscape clearly, cite common protections, and outline what steps a tenant typically takes in this situation.
Always note this is general information, not legal advice.

Question: {legal_query}"""

# Structured reasoning
sr_legal = f"""You are a legal information assistant.
Think through applicable laws and typical processes, then provide:
Relevant Law or Right:
Typical Steps to Take (numbered):
When to Consult an Attorney:
Disclaimer:

Question: {legal_query}"""

# Output format — useful for legal intake tools or FAQ databases
of_legal = f"""You are a legal information assistant.
Return ONLY valid JSON:
{{
  "relevant_rights": [],
  "recommended_steps": [],
  "timeline": "",
  "escalation_path": "",
  "disclaimer": ""
}}

Question: {legal_query}"""

print("=== Legal: Zero-Shot ===")
print(llm.invoke(zs_legal).content[:400])
print("\n=== Legal: Few-Shot ===")
print(llm.invoke(fs_legal).content[:400])
print("\n=== Legal: Role-Based ===")
print(llm.invoke(rb_legal).content[:400])
print("\n=== Legal: Structured Reasoning ===")
print(llm.invoke(sr_legal).content[:400])
print("\n=== Legal: Output Format (JSON) ===")
print(llm.invoke(of_legal).content[:400])

### Domain: Human Resources — HR Assistant

In [ ]:
hr_query = "How do I write a job description that attracts diverse candidates?"

# Zero-shot
zs_hr = f"""You are an HR specialist.
Answer the question with practical, evidence-based guidance.

Question: {hr_query}"""

# Few-shot: examples show inclusive JD best practices
fs_hr = f"""You are an HR specialist focused on inclusive hiring practices.

Example 1:
Question: "What words in job descriptions deter women from applying?"
Answer: Research shows words like "ninja", "rockstar", "dominant", and "competitive" are coded masculine and reduce female applicants. Prefer neutral terms: "collaborative", "skilled", "motivated". Tools like Textio or Gender Decoder can audit your JDs automatically.

Example 2:
Question: "Should I list years of experience as a requirement?"
Answer: Years-of-experience requirements often screen out qualified career-changers and underrepresented candidates. Replace "5+ years experience" with specific skills or competencies (e.g., "proficiency in SQL and dashboard reporting") to widen the qualified pool.

Question: {hr_query}
Answer:"""

# Role-based
rb_hr = f"""You are a DEI-focused talent acquisition leader at a Fortune 500 company.
Give concrete, research-backed advice that a hiring manager can act on today.

Question: {hr_query}"""

# Structured reasoning
sr_hr = f"""You are an HR specialist.
Think through language, structure, and sourcing, then provide:
Key Principles (3 bullet points):
Specific Wording Tips:
Common Mistakes to Avoid:

Question: {hr_query}"""

# Output format — useful for JD template generators
of_hr = f"""You are an HR specialist.
Return ONLY valid JSON for a JD writing checklist:
{{
  "inclusive_language_tips": [],
  "words_to_avoid": [],
  "must_have_sections": [],
  "optional_sections": [],
  "sourcing_channels": []
}}

Question: {hr_query}"""

print("=== HR: Zero-Shot ===")
print(llm.invoke(zs_hr).content[:400])
print("\n=== HR: Few-Shot ===")
print(llm.invoke(fs_hr).content[:400])
print("\n=== HR: Role-Based ===")
print(llm.invoke(rb_hr).content[:400])
print("\n=== HR: Structured Reasoning ===")
print(llm.invoke(sr_hr).content[:400])
print("\n=== HR: Output Format (JSON) ===")
print(llm.invoke(of_hr).content[:400])

### Domain: Real Estate — Property Advisor

In [ ]:
realestate_query = "I'm a first-time buyer with a $400,000 budget. What should I look for when visiting a property?"

# Zero-shot
zs_re = f"""You are a real estate advisor.
Give practical, buyer-focused advice.

Question: {realestate_query}"""

# Few-shot: examples anchor inspection mindset
fs_re = f"""You are a real estate advisor helping first-time buyers make confident decisions.

Example 1:
Question: "What questions should I ask the seller's agent at an open house?"
Answer: Ask about the age of the roof, HVAC, and water heater; any known structural issues; how long the property has been on the market; and whether there are any pending HOA special assessments. These reveal costs you'll inherit as the new owner.

Example 2:
Question: "What's the difference between a home inspection and an appraisal?"
Answer: An inspection assesses the physical condition of the property for the buyer's benefit — a trained inspector checks structure, systems, and safety. An appraisal is ordered by the lender to confirm the property's market value matches the loan amount. You need both.

Question: {realestate_query}
Answer:"""

# Role-based
rb_re = f"""You are a buyer's agent with 20 years of experience in residential real estate.
Think like someone who has seen every hidden problem. Be thorough and protect the buyer.

Question: {realestate_query}"""

# Structured reasoning
sr_re = f"""You are a real estate advisor.
Think through structural, legal, financial, and neighbourhood factors, then provide:
Must-Check Items (numbered):
Red Flags to Walk Away From:
Questions to Ask the Agent:

Question: {realestate_query}"""

# Output format — useful for a property viewing checklist app
of_re = f"""You are a real estate advisor.
Return ONLY valid JSON:
{{
  "structural_checks": [],
  "system_checks": [],
  "neighbourhood_checks": [],
  "questions_for_agent": [],
  "red_flags": []
}}

Question: {realestate_query}"""

print("=== Real Estate: Zero-Shot ===")
print(llm.invoke(zs_re).content[:400])
print("\n=== Real Estate: Few-Shot ===")
print(llm.invoke(fs_re).content[:400])
print("\n=== Real Estate: Role-Based ===")
print(llm.invoke(rb_re).content[:400])
print("\n=== Real Estate: Structured Reasoning ===")
print(llm.invoke(sr_re).content[:400])
print("\n=== Real Estate: Output Format (JSON) ===")
print(llm.invoke(of_re).content[:400])

### Domain: Travel & Hospitality — Travel Planner

In [ ]:
travel_query = "Plan a 5-day budget trip to Japan for a solo traveler in April."

# Zero-shot
zs_travel = f"""You are a travel planner.
Create a practical, budget-conscious itinerary.

Request: {travel_query}"""

# Few-shot: examples show structure and level of detail expected
fs_travel = f"""You are an expert travel planner specializing in budget solo travel.

Example 1:
Request: "Plan a 3-day budget trip to Bangkok."
Answer:
Day 1: Arrive, stay in a hostel in the Banglamphu area (~$12/night). Visit the Grand Palace and Wat Pho (entry ~$15 combined).
Day 2: Chatuchak Weekend Market (free entry), street food lunch ($2–3), Khao San Road in the evening.
Day 3: Day trip to Ayutthaya by train ($2 each way). Return and depart.
Budget estimate: ~$40–50/day including accommodation, food, and transport.

Example 2:
Request: "Plan a 2-day budget trip to Lisbon."
Answer:
Day 1: Alfama walking tour (free), Pastéis de Belém for lunch (~€4), LX Factory market.
Day 2: Sintra day trip by train (~€4), Pena Palace (~€15), return for evening fado.
Budget estimate: ~€50–60/day.

Request: {travel_query}
Itinerary:"""

# Role-based
rb_travel = f"""You are a solo travel specialist who has backpacked through Japan three times.
Write like a friend who knows all the tricks — rail passes, local food, free temples, crowd-avoiding tips.
Include a rough daily budget in USD.

Request: {travel_query}"""

# Structured reasoning
sr_travel = f"""You are a travel planner.
Think through transport, accommodation, must-see vs skip, and budget trade-offs, then provide:
Day-by-Day Itinerary:
Budget Breakdown (per day):
Money-Saving Tips (3):

Request: {travel_query}"""

# Output format — useful for travel app or itinerary builder
of_travel = f"""You are a travel planner.
Return ONLY valid JSON:
{{
  "destination": "",
  "duration_days": 0,
  "daily_budget_usd": 0,
  "itinerary": [
    {{"day": 1, "city": "", "activities": [], "accommodation": "", "food_tip": ""}}
  ],
  "packing_essentials": [],
  "money_saving_tips": []
}}

Request: {travel_query}"""

print("=== Travel: Zero-Shot ===")
print(llm.invoke(zs_travel).content[:500])
print("\n=== Travel: Few-Shot ===")
print(llm.invoke(fs_travel).content[:500])
print("\n=== Travel: Role-Based ===")
print(llm.invoke(rb_travel).content[:500])
print("\n=== Travel: Structured Reasoning ===")
print(llm.invoke(sr_travel).content[:500])
print("\n=== Travel: Output Format (JSON) ===")
print(llm.invoke(of_travel).content[:500])

### Domain: IT Support — Technical Help Desk

In [ ]:
it_query = "My laptop is running very slowly after the latest Windows update. How do I fix it?"

# Zero-shot
zs_it = f"""You are an IT support specialist.
Give step-by-step troubleshooting guidance.

Issue: {it_query}"""

# Few-shot: examples show diagnostic → solution pattern
fs_it = f"""You are an IT support specialist. Always follow a diagnostic-first approach.

Example 1:
Issue: "My Wi-Fi keeps disconnecting every few minutes."
Answer:
1. Update your network adapter driver (Device Manager → Network Adapters → right-click → Update driver).
2. Disable power management for the adapter (Properties → Power Management → uncheck "Allow the computer to turn off this device to save power").
3. If still failing, forget and re-add the Wi-Fi network.
4. Test with a wired connection to isolate hardware vs software.

Example 2:
Issue: "My second monitor is not being detected."
Answer:
1. Check the physical cable connection (try a different cable/port).
2. Right-click Desktop → Display Settings → Detect.
3. Update your GPU driver from the manufacturer's site.
4. Reboot with the monitor connected.

Issue: {it_query}
Answer:"""

# Role-based
rb_it = f"""You are a Level 2 IT support engineer who explains technical fixes in plain language for non-technical users.
Prioritize safe, reversible steps first. Flag anything that requires admin rights.

Issue: {it_query}"""

# Structured reasoning — systematic diagnostic process
sr_it = f"""You are an IT support specialist.
Think through likely causes (resource usage, startup items, driver conflicts, background processes), then provide:
Most Likely Cause:
Step-by-Step Fix (numbered, safest first):
Prevention Tip:

Issue: {it_query}"""

# Output format — useful for ticket system or knowledge base
of_it = f"""You are an IT support specialist.
Return ONLY valid JSON for a support ticket resolution:
{{
  "issue_category": "",
  "likely_causes": [],
  "resolution_steps": [],
  "requires_admin": true,
  "estimated_fix_time_minutes": 0,
  "escalate_if": ""
}}

Issue: {it_query}"""

print("=== IT Support: Zero-Shot ===")
print(llm.invoke(zs_it).content[:400])
print("\n=== IT Support: Few-Shot ===")
print(llm.invoke(fs_it).content[:400])
print("\n=== IT Support: Role-Based ===")
print(llm.invoke(rb_it).content[:400])
print("\n=== IT Support: Structured Reasoning ===")
print(llm.invoke(sr_it).content[:400])
print("\n=== IT Support: Output Format (JSON) ===")
print(llm.invoke(of_it).content[:400])

---
## Multi-Domain Test Harness

Run the same structured test harness across all 10 domains. This validates that prompting patterns generalise beyond a single scenario and lets learners from any background benchmark their domain.

Each domain has 10 diverse inputs covering: normal requests, edge cases, ambiguous queries, and missing-information scenarios.

In [ ]:
# 10 diverse inputs per domain — 10 domains total
domain_inputs = {

    "ecommerce": [
        "My order #12345 was supposed to arrive yesterday. Can you check the status?",
        "I received a damaged mug. How do I get a replacement?",
        "I want to return a jacket bought last week. What's the return process?",
        "I was charged twice for my order. Please help.",
        "I need to change the shipping address for order #98765.",
        "The promo code SPRING20 isn't working at checkout.",
        "Can I cancel my order? It hasn't shipped yet.",
        "I forgot my password and can't log in.",
        "Do you ship internationally to Canada?",
        "I need an invoice for my last purchase.",
    ],

    "finance": [
        "I earn $45,000/year and have $8,000 in credit card debt. Where do I start?",
        "Should I invest in stocks or pay off my student loan first?",
        "What is the difference between a Roth IRA and a Traditional IRA?",
        "I have $500/month to save. What's the best strategy?",
        "My credit score dropped 50 points overnight. Why?",
        "How do I calculate how much house I can afford?",
        "Is it a good time to buy gold right now?",
        "Can I retire at 55 if I save $1,500/month starting at 30?",
        "What does APR mean and how does it affect my loan?",
        "I lost my job. How do I manage my finances for the next 3 months?",
    ],

    "marketing": [
        "Write a tagline for a productivity app aimed at remote workers.",
        "Create a subject line for a back-to-school email campaign.",
        "Suggest 3 hashtags for a sustainable fashion brand post.",
        "Write a 30-second radio ad script for a local bakery.",
        "How should I position a premium gym membership vs. budget competitors?",
        "Write a LinkedIn post announcing a company's 10th anniversary.",
        "Create a call-to-action for a landing page selling an online cooking course.",
        "Write a 2-sentence Instagram caption for a travel bag launch.",
        "What is the best day and time to post on LinkedIn for B2B audiences?",
        "Write a re-engagement email subject line for inactive subscribers.",
    ],

    "education": [
        "Explain the concept of fractions to a 9-year-old.",
        "What is the difference between mitosis and meiosis?",
        "Create 3 multiple-choice quiz questions on the causes of World War I.",
        "Explain Newton's second law with a real-world example.",
        "How would you teach a struggling student to understand algebra?",
        "Summarize the French Revolution in 5 bullet points.",
        "What are the best study techniques backed by cognitive science?",
        "Write a lesson plan outline for teaching climate change to high schoolers.",
        "Explain machine learning to a 12-year-old using a toy analogy.",
        "What is the Socratic method and when should a teacher use it?",
    ],

    "healthcare": [
        "I've had a headache for 3 days. Should I be worried?",
        "What is the difference between Type 1 and Type 2 diabetes?",
        "How do I know if a cut needs stitches?",
        "I'm feeling anxious all the time. What can I do?",
        "What are the symptoms of iron deficiency anemia?",
        "Is it safe to take ibuprofen every day for back pain?",
        "How much sleep does an adult actually need?",
        "What should I do if I think I'm having an allergic reaction?",
        "What are the early warning signs of a stroke?",
        "Can stress cause physical symptoms like chest pain?",
    ],

    "legal": [
        "My landlord hasn't returned my security deposit after 45 days.",
        "My employer is asking me to sign a non-compete agreement. Is it enforceable?",
        "I received a cease-and-desist letter. What should I do?",
        "Can I record a conversation without the other person's consent?",
        "What is the difference between a will and a living trust?",
        "My neighbor's tree fell on my fence. Who pays for repairs?",
        "I was in a minor car accident. Do I need a lawyer?",
        "What rights do I have if I'm fired without notice?",
        "Can my employer monitor my work email?",
        "What is the statute of limitations for small claims court?",
    ],

    "hr": [
        "How do I write a job description that attracts diverse candidates?",
        "An employee reported a harassment complaint. What are the next steps?",
        "How should I structure a performance improvement plan?",
        "What is the legal difference between an employee and a contractor?",
        "How do I conduct a fair and effective performance review?",
        "Our remote employees feel disconnected. What can HR do?",
        "How do I handle a situation where two employees have a personal conflict?",
        "What should be included in an onboarding plan for new hires?",
        "How do we reduce voluntary turnover in a competitive market?",
        "What are best practices for conducting exit interviews?",
    ],

    "realestate": [
        "I'm a first-time buyer with a $400,000 budget. What should I look for?",
        "What is the difference between a fixed-rate and adjustable-rate mortgage?",
        "How do I make a competitive offer in a seller's market?",
        "What fees should I expect when closing on a house?",
        "Is it better to buy or rent in a high-cost city right now?",
        "What does 'as-is' mean when buying a property?",
        "How do I evaluate whether an investment property has good ROI?",
        "My home inspection found a cracked foundation. Should I walk away?",
        "What is earnest money and can I get it back if I back out?",
        "How long does the home buying process typically take?",
    ],

    "travel": [
        "Plan a 5-day budget trip to Japan for a solo traveler in April.",
        "What documents do I need to travel to Europe as a US citizen?",
        "What is travel insurance and do I really need it?",
        "How do I avoid tourist traps in popular destinations?",
        "What are the best apps for finding cheap flights?",
        "Is it safe for a solo female traveler to visit Morocco?",
        "How do I pack a carry-on for a 2-week trip?",
        "What should I do if my flight gets cancelled?",
        "What vaccinations do I need before traveling to Southeast Asia?",
        "How do I find authentic local food experiences when traveling?",
    ],

    "it_support": [
        "My laptop is running very slowly after the latest Windows update.",
        "I accidentally deleted an important file. Can I recover it?",
        "My printer is not connecting to my laptop over Wi-Fi.",
        "How do I set up two-factor authentication on my accounts?",
        "My Outlook keeps crashing when I open large attachments.",
        "I think my computer has a virus. What should I do?",
        "How do I transfer all my files to a new computer?",
        "My screen is flickering randomly. What could cause this?",
        "My keyboard is typing the wrong characters.",
        "How do I reset my Windows password if I've forgotten it?",
    ],
}

print("Domain inputs loaded:", {k: len(v) for k, v in domain_inputs.items()})
print("Total domains:", len(domain_inputs))

In [ ]:
# Domain-specific prompt builder — structured v3-style prompt for all 10 domains
def build_domain_prompt(domain, input_text):
    templates = {
        "ecommerce": """You are a customer support assistant for an e-commerce store.
Guidelines: be empathetic and concise (max 3 sentences). Acknowledge the issue. Provide one actionable next step. Ask for order number if order-related.

Customer message: {input}

Response:
Next Step:""",

        "finance": """You are a certified personal finance advisor.
Guidelines: acknowledge the situation, give jargon-free practical advice, recommend 1–2 concrete steps.

Customer question: {input}

Assessment:
Action Steps:""",

        "marketing": """You are a senior marketing copywriter.
Guidelines: be concise and impactful, use active voice, no buzzwords, include a clear call-to-action where relevant.

Task: {input}

Output:""",

        "education": """You are a subject tutor.
Guidelines: use simple language appropriate to the audience, include an analogy or example, be encouraging.

Task: {input}

Explanation:""",

        "healthcare": """You are a healthcare information assistant.
Guidelines: provide general health information only, list red-flag symptoms that need urgent care, always recommend consulting a licensed doctor. Never diagnose.

Question: {input}

General Information:
When to See a Doctor:
Disclaimer: This is general information, not medical advice.""",

        "legal": """You are a legal FAQ assistant.
Guidelines: explain the general legal landscape clearly, outline typical steps, always recommend consulting a licensed attorney for specific cases.

Question: {input}

General Information:
Typical Steps:
Disclaimer: This is general information, not legal advice.""",

        "hr": """You are an HR specialist.
Guidelines: give evidence-based, actionable advice; consider compliance, fairness, and employee wellbeing.

Question: {input}

Recommendation:
Action Steps:""",

        "realestate": """You are a real estate advisor.
Guidelines: give buyer/seller-focused practical advice, flag important risks, suggest next steps.

Question: {input}

Advice:
Next Steps:""",

        "travel": """You are a travel planner.
Guidelines: be practical and budget-conscious, include specific tips, tailor advice to the traveler's context.

Request: {input}

Plan:
Tips:""",

        "it_support": """You are an IT support specialist.
Guidelines: give numbered diagnostic steps, safest/simplest steps first, flag if admin rights are required.

Issue: {input}

Diagnosis:
Steps to Fix:""",
    }
    template = templates.get(domain, "You are a helpful assistant.\n\n{input}")
    return template.replace("{input}", input_text)


def llm_judge_domain(domain, input_text, output_text):
    judge_prompt = f"""Evaluate this AI response for a {domain} assistant.
Score from 1–10 on: correctness, clarity, and helpfulness. Return only a number.

Input: {input_text}
Response: {output_text}"""
    return llm.invoke(judge_prompt).content.strip()


def run_domain_harness(domain, inputs):
    results = []
    for inp in inputs:
        prompt = build_domain_prompt(domain, inp)
        output = llm.invoke(prompt).content
        score = llm_judge_domain(domain, inp, output)
        results.append({"domain": domain, "input": inp, "output": output, "score": score})
    return results


# Spot-check all 10 domains (first 2 inputs each — remove [:2] for full 10-input run)
for domain, inputs in domain_inputs.items():
    spot = run_domain_harness(domain, inputs[:2])
    avg = sum(float(r["score"]) for r in spot) / len(spot)
    print(f"{domain:15} | spot-check avg: {avg:.1f}")

In [ ]:
# Full multi-domain harness — all 10 inputs per domain
# This takes ~3–4 minutes. Comment out domains you don't need.

all_domain_results = {}

for domain, inputs in domain_inputs.items():
    print(f"Running {domain}...")
    all_domain_results[domain] = run_domain_harness(domain, inputs)

print("\n=== Multi-Domain Score Summary ===")
for domain, results in all_domain_results.items():
    scores = [float(r["score"]) for r in results]
    avg = sum(scores) / len(scores)
    failures = sum(1 for s in scores if s < 8.5)
    print(f"{domain:12} | avg: {avg:.2f} | failures (<8.5): {failures}")

---
## Lab: Structured Drills — Full Scenario

**Domain scenario:** You are building an AI assistant for your domain. Run each prompting pattern on your domain's test inputs, record scores, and document the best-performing pattern.

**Instructions for learners:**
1. Pick your domain from the list below and set `CHOSEN_DOMAIN`
2. Run `lab_drill(domain, domain_inputs[domain])` to get a ranked score table
3. Fill in the `lab_doc` dictionary with your observations
4. Compare your results with classmates who chose different domains

**Available domains:** `ecommerce` · `finance` · `marketing` · `education` · `healthcare` · `legal` · `hr` · `realestate` · `travel` · `it_support`

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

def lab_drill(domain, inputs):
    """Run all 6 prompting patterns on a domain's inputs and return a scored comparison table."""

    def build(pattern, inp):
        if pattern == "zero_shot":
            return f"You are an assistant for {domain}.\nRespond to the request.\n\n{inp}"
        elif pattern == "few_shot":
            return f"You are an assistant for {domain}.\nExample: 'What is X?' → 'X is...'\n\n{inp}"
        elif pattern == "role_based":
            return f"You are a senior {domain} expert known for clarity and accuracy.\n\n{inp}"
        elif pattern == "system_prompt":
            return [
                SystemMessage(content=f"You are a helpful {domain} assistant. Be concise and accurate."),
                HumanMessage(content=inp),
            ]
        elif pattern == "structured_reasoning":
            return f"Think step-by-step, then provide:\nFinal Answer:\nRationale (1 sentence):\n\n{inp}"
        elif pattern == "output_format":
            return f'Return ONLY valid JSON: {{"answer": "", "confidence": "", "next_step": ""}}\n\n{inp}'

    patterns = ["zero_shot", "few_shot", "role_based", "system_prompt", "structured_reasoning", "output_format"]
    scores_by_pattern = {p: [] for p in patterns}

    for inp in inputs:
        for pattern in patterns:
            prompt = build(pattern, inp)
            output = llm.invoke(prompt).content
            score = llm_judge_domain(domain, inp, output)
            scores_by_pattern[pattern].append(float(score))

    summary = [(p, round(sum(s) / len(s), 2)) for p, s in scores_by_pattern.items()]
    summary.sort(key=lambda x: x[1], reverse=True)
    return summary


# Change domain to try a different one: "finance", "marketing", "education"
CHOSEN_DOMAIN = "ecommerce"
drill_results = lab_drill(CHOSEN_DOMAIN, domain_inputs[CHOSEN_DOMAIN])

print(f"\n=== Lab Drill Results: {CHOSEN_DOMAIN} ===")
print(f"{'Pattern':<25} | Avg Score")
print("-" * 38)
for pattern, score in drill_results:
    marker = " ← BEST" if pattern == drill_results[0][0] else ""
    print(f"{pattern:<25} | {score}{marker}")

### Lab Documentation Template

After running the drills, fill in the template below to document your findings. This creates a reproducible record of prompt improvement decisions.

In [ ]:
lab_doc = {
    "learner_name": "Your Name",
    "domain": CHOSEN_DOMAIN,

    # From drill_results above
    "pattern_scores": {pattern: score for pattern, score in drill_results},

    "best_pattern": drill_results[0][0],
    "best_score": drill_results[0][1],

    # Fill these in after reviewing outputs
    "why_best_pattern_works": (
        # Example: "Structured reasoning forced the model to break down ambiguous requests before "
        # "answering, which avoided vague responses on inputs that lacked context."
        "TODO: explain in 1-2 sentences"
    ),

    "worst_pattern": drill_results[-1][0],
    "why_worst_failed": "TODO: describe the failure mode",

    "most_interesting_failure": {
        "input": "TODO: paste the input that failed most",
        "pattern": "TODO: which pattern failed",
        "failure_type": "TODO: wrong format / wrong tone / missing info / hallucination / too verbose",
        "fix_applied": "TODO: what change to the prompt fixed it",
    },

    "prompt_improvement_log": [
        {
            "version": "v1",
            "change": "baseline — minimal instruction",
            "avg_score": None,   # fill after running
            "failures": None,
        },
        {
            "version": "v2",
            "change": "TODO: describe the change you made",
            "avg_score": None,
            "failures": None,
        },
    ],

    "key_takeaway": "TODO: one sentence on what you learned about prompting in this domain",
}

# Pretty print the documentation
import json
print(json.dumps(lab_doc, indent=2))

---
## Summary: What We Covered

### Prompting Patterns

| Pattern | Key Concept | Takeaway |
|---------|-------------|----------|
| Zero-shot | No examples given | Works for simple tasks; fails when format or tone matter |
| Few-shot | 2–3 examples in prompt | Anchors format, vocabulary, and tone; reduces ambiguity |
| Role-based | "You are X..." | Shifts style, depth, and reasoning approach |
| System prompt | Separate system/user messages | Cleaner structure; more reliable across multi-turn sessions |
| Structured reasoning | Step-by-step before answering | Improves accuracy on multi-step and diagnostic problems |
| Output format control | JSON / list / structured text | Essential for downstream processing and integrations |
| Self-checking | Model critiques its own answer | Catches factual gaps; improves completeness |
| Iterative refinement | Multi-pass improvement loop | Simulates deeper thinking; use judiciously (costs tokens) |

### Prompt Engineering Process

| Step | Action | Output |
|------|--------|--------|
| Prompt versioning | v1 → v2 → v3, change one thing at a time | Evidence-based improvement log |
| Test harness | 10 diverse inputs + LLM judge | Prevents overfitting to a single example |
| Failure analysis | Score threshold + manual review | Reveals weaknesses before deployment |

### Domain Coverage

| Domain | What it tests |
|--------|--------------|
| E-commerce | Empathy, structured next steps, order context |
| Finance | Jargon-free advice, numerical reasoning, risk flagging |
| Marketing | Creativity, conciseness, audience awareness |
| Education | Analogy-building, vocabulary calibration, scaffolding |
| Healthcare | Safety guardrails, triage logic, disclaimer handling |
| Legal | Boundary-setting, general vs. specific advice distinction |
| HR | DEI sensitivity, compliance awareness, policy tone |
| Real Estate | Risk awareness, procedural knowledge, buyer/seller POV |
| Travel | Contextual planning, budget trade-offs, practical logistics |
| IT Support | Diagnostic reasoning, step prioritisation, non-technical language |

**Next steps:** Pick a domain, run `lab_drill()`, iterate to a `v4` prompt, and document what changed and why.